In [1]:
import great_expectations as gx
import pandas as pd

df = pd.read_excel("customer_data.xlsx")
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="pd dataframe asset")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})


In [2]:
suite = gx.ExpectationSuite(name="customer_data_expectations")
suite = context.suites.add(suite)


# customer_id — unique AND not null (two separate expectations)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column="customer_id")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")
)

# age — between 0 and 120
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="age", min_value=0, max_value=120
    )
)

# email — valid email format via regex
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="email",
        regex=r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$",
    )
)

# salary — present in at least 95% of rows (mostly handles the threshold)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="salary", mostly=0.95)
)

# country — one of the allowed values
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="country", value_set=["USA", "Canada", "UK", "Australia"]
    )
)

# signup_date — datetime type
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="signup_date", type_="datetime64[ns]"
    )
)

# whole-table row count — between 500 and 1000
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=500, max_value=1000)
)

results = batch.validate(suite)

print("Overall success:", results.success)
print(results)

Calculating Metrics:   0%|          | 0/42 [00:00<?, ?it/s]

Overall success: False
{
  "success": false,
  "results": [
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "pandas-pd dataframe asset",
          "column": "customer_id"
        },
        "meta": {},
        "id": "7dd5d021-036f-4d5f-a50c-3666ab5fd8c7",
        "severity": "critical"
      },
      "result": {
        "element_count": 5015,
        "unexpected_count": 568,
        "unexpected_percent": 11.675231243576567,
        "partial_unexpected_list": [
          "C00923",
          "C01991",
          "C00380",
          "C00801",
          "C00113",
          "C00019",
          "C02854",
          "C00822",
          "C00834",
          "C00512",
          "C00804",
          "C00107",
          "C00100",
          "C00200",
          "C00157",
          "C00191",
          "C00350",
          "C00009",
          "C00562",
          "C01060"
        ],
        "missing_c

In [3]:
from great_expectations.checkpoint import UpdateDataDocsAction

# Link the batch + suite into a validation definition
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_definition,
        suite=suite,
        name="customer_validation_def",
    )
)

# A checkpoint runs the validation and updates Data Docs
checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="customer_checkpoint",
        validation_definitions=[validation_definition],
        actions=[UpdateDataDocsAction(name="update_data_docs")],
    )
)

checkpoint.run(batch_parameters={"dataframe": df})
context.open_data_docs()   

Calculating Metrics:   0%|          | 0/42 [00:00<?, ?it/s]